# Week 16: Advanced MLOps & Distributed Deep Learning
## Notebook 02: Tensor Parallelism & Pipeline Parallelism (Megatron-LM)

### 1. Conceptual Intuition & Real-World Use Case
When training 70B+ or 405B+ parameter LLMs (like LLaMA-3), a single model weight tensor (e.g. MLP intermediate layer $4096 \times 14336$) exceeds several gigabytes and cannot fit on a single GPU.

To solve this, Megatron-LM introduced:
1. **Tensor Parallelism (TP):** Slices individual weight matrices across multiple GPUs along rows or columns within the same server (connected by high-speed NVLink).
2. **Pipeline Parallelism (PP):** Slices different transformer layers across different servers (e.g. Layers 1-8 on Node 1, Layers 9-16 on Node 2).

---

### 2. Mathematical Foundations
In an MLP block: $\text{MLP}(X) = \text{GELU}(X W_1) W_2$
Where $X \in \mathbb{R}^{B \times H}$, $W_1 \in \mathbb{R}^{H \times 4H}$, $W_2 \in \mathbb{R}^{4H \times H}$.

#### Megatron-LM 1D Tensor Slicing:
1. **Column-Parallel Linear Layer ($W_1$):**
   Split $W_1$ column-wise across 2 GPUs:
   $$W_1 = [W_{1,1} \parallel W_{1,2}]$$
   Each GPU computes:
   $$Y_1 = \text{GELU}(X W_{1,1}), \quad Y_2 = \text{GELU}(X W_{1,2})$$
   *Zero communication required here!*

2. **Row-Parallel Linear Layer ($W_2$):**
   Split $W_2$ row-wise:
   $$W_2 = \begin{bmatrix} W_{2,1} \\ W_{2,2} \end{bmatrix}$$
   Each GPU computes:
   $$Z_1 = Y_1 W_{2,1}, \quad Z_2 = Y_2 W_{2,2}$$
   Notice:
   $$Z_1 + Z_2 = Y_1 W_{2,1} + Y_2 W_{2,2} = [Y_1 \parallel Y_2] \begin{bmatrix} W_{2,1} \\ W_{2,2} \end{bmatrix} = \text{GELU}(X W_1) W_2$$
   **Exactly ONE All-Reduce Sum operation** is needed across the GPUs at the very end of the MLP block!

#### Pipeline Bubble Fraction:
In naive pipeline parallelism (GPipe), GPU utilization suffers from idle "bubble" time:
$$\text{Bubble Fraction} = \frac{p - 1}{m + p - 1}$$
where $p$ is pipeline stages and $m$ is micro-batches. **1F1B (One Forward, One Backward)** interleaves micro-batches to minimize peak memory.

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Simulate Monolithic MLP on Single Device
B = 2      # Batch size
H = 4      # Hidden dimension
inter = 8  # Intermediate dimension (4 * H)

X = np.random.randn(B, H)
W1 = np.random.randn(H, inter)
W2 = np.random.randn(inter, H)

def gelu(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * np.power(x, 3))))

# Monolithic Forward Pass
H_act = gelu(X @ W1)
Y_monolithic = H_act @ W2
print("Monolithic MLP Output (Batch 2 x Hidden 4):")
print(np.round(Y_monolithic, 4))

# Step 2: Megatron-LM 1D Tensor Parallelism across 2 GPUs
# GPU 0 gets first half of intermediate dimension; GPU 1 gets second half
W1_gpu0 = W1[:, :inter // 2]  # Column split (4, 4)
W1_gpu1 = W1[:, inter // 2:]  # Column split (4, 4)

W2_gpu0 = W2[:inter // 2, :]  # Row split (4, 4)
W2_gpu1 = W2[inter // 2:, :]  # Row split (4, 4)

print(f"\nW1 split: GPU 0 shape {W1_gpu0.shape}, GPU 1 shape {W1_gpu1.shape}")
print(f"W2 split: GPU 0 shape {W2_gpu0.shape}, GPU 1 shape {W2_gpu1.shape}")

# GPU 0 local forward pass (no inter-GPU communication!)
Y1_gpu0 = gelu(X @ W1_gpu0)
Z_gpu0 = Y1_gpu0 @ W2_gpu0

# GPU 1 local forward pass (no inter-GPU communication!)
Y1_gpu1 = gelu(X @ W1_gpu1)
Z_gpu1 = Y1_gpu1 @ W2_gpu1

# All-Reduce Sum across GPUs
Y_parallel = Z_gpu0 + Z_gpu1

print("\nTensor Parallel Combined Output:")
print(np.round(Y_parallel, 4))

diff = np.max(np.abs(Y_monolithic - Y_parallel))
print(f"\nMax Absolute Difference vs Monolithic: {diff:.2e}")
print("Exact Match:", np.allclose(Y_monolithic, Y_parallel))

### 3. Step-by-Step Code Walkthrough
1. **Column Parallel ($W_1$):** Splitting columns means neither GPU needs communication before applying non-linear activation (GELU).
2. **Row Parallel ($W_2$):** Splitting rows naturally aligns matrix inner dimensions.
3. **Single All-Reduce:** Only one communication step (`Z_gpu0 + Z_gpu1`) is required at the end, exactly replicating full model forward pass to floating-point precision.

---

### 4. Industry Applications & Trade-offs
- **NVLink Requirement:** Tensor Parallelism requires ultra-high bandwidth (900 GB/s on H100s). It is typically confined within a single 8-GPU node.
- **Pipeline Parallelism:** Used across separate physical server racks connected via slower Infiniband (400 Gbps).

---

### 5. Top Interview Questions & Answers
**Q1: Why cannot Tensor Parallelism easily scale across different physical server nodes?**
> **Answer:** TP triggers AllReduce communication after every single Transformer layer. Across nodes, network latency (TCP/Ethernet) causes severe GPU stalling. Thus, TP is restricted to intra-node NVLink (usually $\le 8$ GPUs).

**Q2: What is the 1F1B schedule in Pipeline Parallelism?**
> **Answer:** In 1F1B (One Forward, One Backward), each GPU executes one forward micro-batch followed immediately by one backward micro-batch. This discards activations as soon as possible, drastically reducing peak memory compared to GPipe which executes all forward passes first.